# Today Replay and PnL (No Orders)

Runs the current `Updated_Bot` scanner logic on FYERS 15m history and computes hypothetical PnL. This notebook does not call broker order functions.

In [ ]:
import os
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd
from IPython.display import display

BOT_DIR = Path(r"E:\ShoonyaAPI_Code\Updated_Bot")
os.chdir(BOT_DIR)
sys.path.insert(0, str(BOT_DIR))

import config
import fyers_data
import state as app_state
import shoonya_scanner_bot as bot

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("BOT_DIR", BOT_DIR)
print("SCAN_TIMES", config.SCAN_TIMES)

## 1. Load universe and FYERS 15m history

In [ ]:
universe = bot.load_universe()
cache = fyers_data.refresh_fyers_cache(
    universe,
    days_back=config.FYERS_HISTORY_DAYS,
    max_workers=config.FYERS_WORKERS,
)

print("UNIVERSE", len(universe))
print("CACHE_SYMBOLS", len(cache))

## 2. Nifty gate and scan replay

In [ ]:
import os
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd
from IPython.display import display

BOT_DIR = Path(r"E:\ShoonyaAPI_Code\Updated_Bot")
os.chdir(BOT_DIR)
sys.path.insert(0, str(BOT_DIR))

import config
import fyers_data
import shoonya_scanner_bot as bot


def fetch_nifty_history():
    end_date = config.now_ist().date()
    start_date = end_date - timedelta(days=config.FYERS_HISTORY_DAYS)

    payload = {
        "symbol": "NSE:NIFTY50-INDEX",
        "resolution": "15",
        "date_format": "1",
        "range_from": start_date.strftime("%Y-%m-%d"),
        "range_to": end_date.strftime("%Y-%m-%d"),
        "cont_flag": "1",
    }

    response = None
    for _ in range(15):
        fyers_data._pace()
        response = fyers_data._client().history(data=payload)
        if fyers_data._rate_limited(response):
            fyers_data._mark_rate_limited()
            continue
        break

    if not isinstance(response, dict) or str(response.get("s", "")).lower() != "ok":
        raise RuntimeError("Nifty history rejected: %s" % response)

    candles = response.get("candles")
    if not isinstance(candles, list) or not candles:
        raise RuntimeError("Nifty history empty")

    frame = pd.DataFrame(candles, columns=fyers_data.HISTORY_COLUMNS[:6])
    raw_times = pd.to_numeric(frame["Timestamp"], errors="coerce")
    largest = raw_times.abs().max()
    unit = "ms" if pd.notna(largest) and largest > 1e11 else "s"

    frame["Timestamp"] = (
        pd.to_datetime(raw_times, unit=unit, utc=True)
        .dt.tz_convert("Asia/Kolkata")
        .dt.tz_localize(None)
    )

    for column in fyers_data.HISTORY_COLUMNS[1:]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    clock = frame["Timestamp"].dt.strftime("%H:%M")
    frame = frame.loc[(clock >= "09:15") & (clock <= "15:15")]
    frame = frame.dropna(subset=["Timestamp", "Close"])
    frame = (
        frame.drop_duplicates(subset=["Timestamp"])
        .sort_values("Timestamp")
        .reset_index(drop=True)
    )

    return frame[fyers_data.HISTORY_COLUMNS]


def nifty_at(bar_timestamp, history):
    if history is not None and not history.empty:
        current = history.loc[history["Timestamp"] <= bar_timestamp]
        if not current.empty:
            current_row = current.iloc[-1]
            previous = history.loc[
                history["Timestamp"] < pd.Timestamp(bar_timestamp).normalize()
            ]

            previous_close = (
                float(previous.iloc[-1]["Close"])
                if not previous.empty
                else float(current_row["Close"])
            )
            ltp = float(current_row["Close"])

            if previous_close > 0:
                percent = ((ltp - previous_close) / previous_close) * 100.0
                return {
                    "ltp": ltp,
                    "previous": previous_close,
                    "percent": percent,
                    "source": "HISTORY",
                }

    quote = fyers_data.get_nifty_quote()
    if not quote:
        raise RuntimeError("No Nifty quote available")

    ltp = float(quote["lp"])
    previous_close = float(quote["c"])
    percent = (
        ((ltp - previous_close) / previous_close) * 100.0
        if previous_close > 0
        else 0.0
    )

    return {
        "ltp": ltp,
        "previous": previous_close,
        "percent": percent,
        "source": "QUOTE",
    }


def short(value, limit=260):
    text = str(value or "")
    return text if len(text) <= limit else text[:limit] + "...[truncated]"


try:
    nifty_history = fetch_nifty_history()
    print("NIFTY_BARS", len(nifty_history))
except Exception as exc:
    nifty_history = None
    print("NIFTY_HISTORY_FALLBACK", exc)


rows = []
for scan_hm in config.SCAN_TIMES:
    bar_timestamp = bot.expected_bar_timestamp(scan_hm)
    nifty = nifty_at(bar_timestamp, nifty_history)
    gate = "BLOCKED" if nifty["percent"] > config.NIFTY_MAX_PCT else "OK"

    matched, reject_summary, evaluated, stale_count, audit = (
        bot.evaluate_candidates(cache, universe, bar_timestamp)
    )

    rows.append({
        "Scan": scan_hm,
        "Bar": bar_timestamp.strftime("%Y-%m-%d %H:%M"),
        "Nifty_LTP": round(nifty["ltp"], 2),
        "Nifty_Prev": round(nifty["previous"], 2),
        "Nifty_Pct": round(nifty["percent"], 4),
        "Gate": gate,
        "Nifty_Source": nifty["source"],
        "Evaluated": evaluated,
        "Stale": stale_count,
        "Match_Count": len(matched),
        "Matches": ",".join(matched) if matched else "NONE",
        "Near": short(audit),
        "Rejects": short(reject_summary),
    })

signals_df = pd.DataFrame(rows)
display(signals_df)

In [ ]:
import sys
print(sys.executable)
%pip install fyers-apiv3

## 3. PnL replay with current bot rules

Entry is approximated at the scan-time 15m open. If SL and TP occur in the same 15m bar, SL is treated first. Charges use `config.CHARGE_PER_TRADE`.

In [ ]:
def calculate_trade(symbol, frame, entry_ts, eod_cut, per_trade_capital):
    if frame is None or frame.empty:
        return None
    entry_row = frame.loc[frame["Timestamp"] == entry_ts]
    if entry_row.empty:
        entry_row = frame.loc[frame["Timestamp"] >= entry_ts].head(1)
    if entry_row.empty:
        return None
    actual_entry_time = pd.Timestamp(entry_row["Timestamp"].iloc[0])
    entry = float(entry_row["Open"].iloc[0])
    if entry <= 0:
        return None
    qty = int(per_trade_capital // entry)
    if qty <= 0:
        return None
    sl = bot.round_tick(entry * (1.0 + config.SL_PCT))
    tp = bot.round_tick(entry * (1.0 - config.TP_PCT))
    post = frame.loc[(frame["Timestamp"] > actual_entry_time) & (frame["Timestamp"] <= eod_cut)]
    exit_price = None
    exit_reason = None
    exit_time = None
    for _, bar in post.iterrows():
        high = float(bar["High"])
        low = float(bar["Low"])
        if high >= sl:
            exit_price = sl
            exit_reason = "SL"
            exit_time = pd.Timestamp(bar["Timestamp"])
            break
        if low <= tp:
            exit_price = tp
            exit_reason = "TP"
            exit_time = pd.Timestamp(bar["Timestamp"])
            break
    if exit_price is None:
        eod_row = frame.loc[frame["Timestamp"] >= eod_cut].head(1)
        if not eod_row.empty:
            exit_price = float(eod_row["Open"].iloc[0])
            exit_reason = "EOD_OPEN"
            exit_time = pd.Timestamp(eod_row["Timestamp"].iloc[0])
        elif not post.empty:
            exit_price = float(post["Close"].iloc[-1])
            exit_reason = "LAST_CLOSE"
            exit_time = pd.Timestamp(post["Timestamp"].iloc[-1])
        else:
            return None
    gross = (entry - exit_price) * qty
    net = gross - config.CHARGE_PER_TRADE
    return {
        "Symbol": symbol,
        "Entry_Time": actual_entry_time,
        "Entry": round(entry, 2),
        "Qty": qty,
        "Deployed": round(qty * entry, 2),
        "SL": sl,
        "TP": tp,
        "Exit_Time": exit_time,
        "Exit": round(exit_price, 2),
        "Exit_Reason": exit_reason,
        "Gross_PnL": round(gross, 2),
        "Charges": config.CHARGE_PER_TRADE,
        "Net_PnL": round(net, 2),
    }


per_trade_capital = app_state.read_capital()
total_capital = 200000.0
first_bar = bot.expected_bar_timestamp(config.SCAN_TIMES[0])
eod_cut = pd.Timestamp("%s %s" % (pd.Timestamp(first_bar).strftime("%Y-%m-%d"), config.EOD_TIME))

open_positions = {}
traded_today = set()
trades = []
skips = []

for row in signals_df.itertuples(index=False):
    scan_now = pd.Timestamp(row.Bar) + pd.Timedelta(minutes=15)
    for symbol in list(open_positions):
        if open_positions[symbol]["Exit_Time"] <= scan_now:
            trades.append(open_positions.pop(symbol))
    if row.Gate == "BLOCKED":
        skips.append({"Scan": row.Scan, "Symbol": "", "Reason": "NIFTY_GATE", "Matches": row.Matches})
        continue
    matches = [] if str(row.Matches) == "NONE" else [item.strip() for item in str(row.Matches).split(",") if item.strip()]
    if len(matches) > config.MAX_SIGNALS_PER_TS:
        skips.append({"Scan": row.Scan, "Symbol": "", "Reason": "CROWD_FILTER", "Matches": row.Matches})
        continue
    for symbol in matches:
        if symbol in open_positions:
            skips.append({"Scan": row.Scan, "Symbol": symbol, "Reason": "ALREADY_OPEN", "Matches": row.Matches})
            continue
        if symbol in traded_today:
            skips.append({"Scan": row.Scan, "Symbol": symbol, "Reason": "ALREADY_TRADED", "Matches": row.Matches})
            continue
        if len(open_positions) >= config.MAX_CONCURRENT:
            skips.append({"Scan": row.Scan, "Symbol": symbol, "Reason": "MAX_POSITIONS", "Matches": row.Matches})
            continue
        trade = calculate_trade(symbol, cache.get(symbol), scan_now, eod_cut, per_trade_capital)
        if trade is None:
            skips.append({"Scan": row.Scan, "Symbol": symbol, "Reason": "NO_DATA_OR_ENTRY", "Matches": row.Matches})
            continue
        open_positions[symbol] = trade
        traded_today.add(symbol)

trades.extend(open_positions.values())
trades_df = pd.DataFrame(trades)
if not trades_df.empty:
    trades_df = trades_df.sort_values("Entry_Time").reset_index(drop=True)
skips_df = pd.DataFrame(skips)

print("PER_TRADE_CAPITAL", per_trade_capital)
print("TOTAL_CAPITAL", total_capital)
display(trades_df)

In [ ]:
display(skips_df)

## 4. Summary and actual bot state

In [ ]:
if trades_df.empty:
    replay_net = 0.0
    replay_gross = 0.0
    replay_deployed = 0.0
    wins = 0
    losses = 0
else:
    replay_net = float(trades_df["Net_PnL"].sum())
    replay_gross = float(trades_df["Gross_PnL"].sum())
    replay_deployed = float(trades_df["Deployed"].sum())
    wins = int((trades_df["Net_PnL"] > 0).sum())
    losses = int((trades_df["Net_PnL"] < 0).sum())

actual_state = app_state.load_state()
actual_trades = app_state.read_csv_rows(config.trade_csv_path(), 500)
actual_pnl = sum(float(row.get("Est_Net_PnL") or 0.0) for row in actual_trades)

summary_df = pd.DataFrame([
    {"Item": "Replay trades", "Value": len(trades_df)},
    {"Item": "Replay wins", "Value": wins},
    {"Item": "Replay losses", "Value": losses},
    {"Item": "Replay gross PnL", "Value": round(replay_gross, 2)},
    {"Item": "Replay charges", "Value": round(len(trades_df) * config.CHARGE_PER_TRADE, 2)},
    {"Item": "Replay net PnL", "Value": round(replay_net, 2)},
    {"Item": "Deployed capital", "Value": round(replay_deployed, 2)},
    {"Item": "Return on 200000", "Value": round((replay_net / total_capital) * 100.0, 4)},
    {"Item": "Actual bot closed trades", "Value": len(actual_trades)},
    {"Item": "Actual bot net PnL", "Value": round(actual_pnl, 2)},
    {"Item": "Actual traded_today", "Value": ",".join(actual_state.get("traded_today", [])) or "NONE"},
    {"Item": "Actual open positions", "Value": len(actual_state.get("open_positions", {}))},
])
display(summary_df)
print("NO_ORDERS_PLACED_BY_NOTEBOOK")